In [0]:
# ============================================================
# CELL 1: Test Serverless Environment
# ============================================================

# Check Spark is working
print(f"Spark version: {spark.version}")
print(f"Python version:")
import sys
print(sys.version)

# Check Delta Lake is available
from delta.tables import DeltaTable
print("Delta Lake imported successfully")

# Check PySpark functions available
from pyspark.sql.functions import col, current_timestamp, lit
print("PySpark functions imported successfully")

print()
print("Environment ready. Let's build.")

Spark version: 4.1.0
Python version:
3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
Delta Lake imported successfully
PySpark functions imported successfully

Environment ready. Let's build.


In [0]:
# ============================================================
# CELL 3: Read All NHS CSV Files Into Bronze DataFrame
# ============================================================

from pyspark.sql.functions import (
    col, lit, current_timestamp
)

# Base path where your files are stored
base_path = "/Workspace/Users/harakrishnashah@gmail.com/"

# Read all CSV files in one shot using wildcard
raw_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("recursiveFileLookup", "true") \
    .csv(f"{base_path}*.csv")

# Add Bronze metadata columns
# Note: Unity Catalog uses _metadata.file_path instead of input_file_name()
bronze_df = raw_df \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .withColumn("_pipeline_version", lit("1.0"))

# Confirm it worked
print(f"Total records ingested: {bronze_df.count()}")
print(f"Total columns: {len(bronze_df.columns)}")
print()
print("Column names:")
for c in bronze_df.columns:
    print(f"  {c}")

Total records ingested: 2382
Total columns: 25

Column names:
  Period
  Org Code
  Parent Org
  Org name
  A&E attendances Type 1
  A&E attendances Type 2
  A&E attendances Other A&E Department
  A&E attendances Booked Appointments Type 1
  A&E attendances Booked Appointments Type 2
  A&E attendances Booked Appointments Other Department
  Attendances over 4hrs Type 1
  Attendances over 4hrs Type 2
  Attendances over 4hrs Other Department
  Attendances over 4hrs Booked Appointments Type 1
  Attendances over 4hrs Booked Appointments Type 2
  Attendances over 4hrs Booked Appointments Other Department
  Patients who have waited 4-12 hs from DTA to admission
  Patients who have waited 12+ hrs from DTA to admission
  Emergency admissions via A&E - Type 1
  Emergency admissions via A&E - Type 2
  Emergency admissions via A&E - Other A&E department
  Other emergency admissions
  _ingestion_timestamp
  _source_file
  _pipeline_version


In [0]:
# ============================================================
# CELL 4: Preview Bronze Data
# ============================================================

print("Sample of raw NHS A&E data:")
print()

# Show first 5 rows
display(bronze_df.limit(5))

print()
print("Data types of each column:")
for field in bronze_df.schema.fields:
    print(f"  {field.name}: {field.dataType}")

Sample of raw NHS A&E data:



Period,Org Code,Parent Org,Org name,A&E attendances Type 1,A&E attendances Type 2,A&E attendances Other A&E Department,A&E attendances Booked Appointments Type 1,A&E attendances Booked Appointments Type 2,A&E attendances Booked Appointments Other Department,Attendances over 4hrs Type 1,Attendances over 4hrs Type 2,Attendances over 4hrs Other Department,Attendances over 4hrs Booked Appointments Type 1,Attendances over 4hrs Booked Appointments Type 2,Attendances over 4hrs Booked Appointments Other Department,Patients who have waited 4-12 hs from DTA to admission,Patients who have waited 12+ hrs from DTA to admission,Emergency admissions via A&E - Type 1,Emergency admissions via A&E - Type 2,Emergency admissions via A&E - Other A&E department,Other emergency admissions,_ingestion_timestamp,_source_file,_pipeline_version
MSitAE-SEPTEMBER-2025,AQN04,NHS ENGLAND SOUTH EAST,PHL LYMINGTON UTC,0,0,2988,0,0,11,0,0,15,0,0,0,0,0,0,0,0,0,2026-06-29T12:37:46.021Z,dbfs:/Workspace/Users/harakrishnashah@gmail.com/2025-09-ae-attendances.csv,1.0
MSitAE-SEPTEMBER-2025,RBQ,NHS ENGLAND NORTH WEST,LIVERPOOL HEART AND CHEST HOSPITAL NHS FOUNDATION TRUST,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,134,2026-06-29T12:37:46.021Z,dbfs:/Workspace/Users/harakrishnashah@gmail.com/2025-09-ae-attendances.csv,1.0
MSitAE-SEPTEMBER-2025,RLQ,NHS ENGLAND MIDLANDS,WYE VALLEY NHS TRUST,5972,1130,155,107,0,0,2510,0,0,26,0,0,443,274,1448,0,0,142,2026-06-29T12:37:46.021Z,dbfs:/Workspace/Users/harakrishnashah@gmail.com/2025-09-ae-attendances.csv,1.0
MSitAE-SEPTEMBER-2025,Y02615,NHS ENGLAND MIDLANDS,SOUTH BIRMINGHAM GP WALK IN CENTRE,0,0,6084,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-06-29T12:37:46.021Z,dbfs:/Workspace/Users/harakrishnashah@gmail.com/2025-09-ae-attendances.csv,1.0
MSitAE-SEPTEMBER-2025,AAH,NHS ENGLAND SOUTH WEST,TETBURY HOSPITAL TRUST LTD,0,0,593,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-06-29T12:37:46.021Z,dbfs:/Workspace/Users/harakrishnashah@gmail.com/2025-09-ae-attendances.csv,1.0



Data types of each column:
  Period: StringType()
  Org Code: StringType()
  Parent Org: StringType()
  Org name: StringType()
  A&E attendances Type 1: StringType()
  A&E attendances Type 2: StringType()
  A&E attendances Other A&E Department: StringType()
  A&E attendances Booked Appointments Type 1: StringType()
  A&E attendances Booked Appointments Type 2: StringType()
  A&E attendances Booked Appointments Other Department: StringType()
  Attendances over 4hrs Type 1: StringType()
  Attendances over 4hrs Type 2: StringType()
  Attendances over 4hrs Other Department: StringType()
  Attendances over 4hrs Booked Appointments Type 1: StringType()
  Attendances over 4hrs Booked Appointments Type 2: StringType()
  Attendances over 4hrs Booked Appointments Other Department: StringType()
  Patients who have waited 4-12 hs from DTA to admission: StringType()
  Patients who have waited 12+ hrs from DTA to admission: StringType()
  Emergency admissions via A&E - Type 1: StringType()
  Emerge

In [0]:
# ============================================================
# CELL 5: Clean Column Names & Write Bronze Delta Table
# ============================================================

import re

def clean_column_name(col_name):
    """
    Clean column names for Delta Lake compatibility:
    - Replace spaces with underscores
    - Remove special characters: & , ; { } ( ) - 
    - Convert to lowercase
    - Remove duplicate underscores
    """
    # Replace spaces and special characters with underscore
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', col_name)
    # Remove duplicate underscores
    clean = re.sub(r'_+', '_', clean)
    # Strip leading/trailing underscores
    clean = clean.strip('_')
    # Convert to lowercase
    clean = clean.lower()
    return clean

# Apply cleaning to all column names
cleaned_df = bronze_df
for col_name in bronze_df.columns:
    new_name = clean_column_name(col_name)
    cleaned_df = cleaned_df.withColumnRenamed(col_name, new_name)

# Show cleaned column names
print("Cleaned column names:")
for c in cleaned_df.columns:
    print(f"  {c}")

print()
print(f"Total columns: {len(cleaned_df.columns)}")

Cleaned column names:
  period
  org_code
  parent_org
  org_name
  a_e_attendances_type_1
  a_e_attendances_type_2
  a_e_attendances_other_a_e_department
  a_e_attendances_booked_appointments_type_1
  a_e_attendances_booked_appointments_type_2
  a_e_attendances_booked_appointments_other_department
  attendances_over_4hrs_type_1
  attendances_over_4hrs_type_2
  attendances_over_4hrs_other_department
  attendances_over_4hrs_booked_appointments_type_1
  attendances_over_4hrs_booked_appointments_type_2
  attendances_over_4hrs_booked_appointments_other_department
  patients_who_have_waited_4_12_hs_from_dta_to_admission
  patients_who_have_waited_12_hrs_from_dta_to_admission
  emergency_admissions_via_a_e_type_1
  emergency_admissions_via_a_e_type_2
  emergency_admissions_via_a_e_other_a_e_department
  other_emergency_admissions
  ingestion_timestamp
  source_file
  pipeline_version

Total columns: 25


In [0]:
# ============================================================
# CELL 6: Write Bronze Delta Table
# ============================================================

bronze_path = "/Workspace/Users/harakrishnashah@gmail.com/bronze/delta/ae_attendances"

cleaned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(bronze_path)

print("Bronze Delta table written successfully")
print()

# Verify
verify_df = spark.read.format("delta").load(bronze_path)
print(f"Records in Delta table: {verify_df.count()}")
print(f"Columns in Delta table: {len(verify_df.columns)}")

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-5209028560330670>, line 11
      1 # ============================================================
      2 # CELL 6: Write Bronze Delta Table
      3 # ============================================================
      5 bronze_path = "/Workspace/Users/harakrishnashah@gmail.com/bronze/delta/ae_attendances"
      7 cleaned_df.write \
      8     .format("delta") \
      9     .mode("overwrite") \
     10     .option("overwriteSchema", "true") \
---> 11     .save(bronze_path)
     13 print("Bronze Delta table written successfully")
     14 print()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_comman

In [0]:
# ============================================================
# CELL 6: Create Unity Catalog Volume for Storage
# ============================================================

# Create catalog, database and volume for our project
spark.sql("CREATE CATALOG IF NOT EXISTS healthcare_pipeline")
spark.sql("USE CATALOG healthcare_pipeline")
spark.sql("CREATE DATABASE IF NOT EXISTS ae_data")
spark.sql("USE DATABASE ae_data")
spark.sql("CREATE VOLUME IF NOT EXISTS bronze_volume")

print("Catalog created: healthcare_pipeline")
print("Database created: ae_data")
print("Volume created: bronze_volume")
print()

# Check the volume path
volume_path = "/Volumes/healthcare_pipeline/ae_data/bronze_volume"
print(f"Write path: {volume_path}")

Catalog created: healthcare_pipeline
Database created: ae_data
Volume created: bronze_volume

Write path: /Volumes/healthcare_pipeline/ae_data/bronze_volume


In [0]:
# ============================================================
# CELL 7: Write Bronze Delta Table to Volume
# ============================================================

bronze_path = "/Volumes/healthcare_pipeline/ae_data/bronze_volume/delta/ae_attendances"

cleaned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(bronze_path)

print("Bronze Delta table written successfully")
print()

# Verify
verify_df = spark.read.format("delta").load(bronze_path)
print(f"Records in Delta table: {verify_df.count()}")
print(f"Columns in Delta table: {len(verify_df.columns)}")
print()
print("Bronze layer complete!")

Bronze Delta table written successfully

Records in Delta table: 2382
Columns in Delta table: 25

Bronze layer complete!


In [0]:
# ============================================================
# CELL 8: Verify Bronze Table & Show Sample Data
# ============================================================

bronze_path = "/Volumes/healthcare_pipeline/ae_data/bronze_volume/delta/ae_attendances"

# Read the Delta table back
bronze_verified = spark.read.format("delta").load(bronze_path)

print("=== BRONZE LAYER VERIFICATION REPORT ===")
print()
print(f"Total records:  {bronze_verified.count()}")
print(f"Total columns:  {len(bronze_verified.columns)}")
print()

# Show sample data
print("Sample data:")
display(bronze_verified.limit(5))

print()
print("Bronze layer verified and complete!")

=== BRONZE LAYER VERIFICATION REPORT ===

Total records:  2382
Total columns:  25

Sample data:


period,org_code,parent_org,org_name,a_e_attendances_type_1,a_e_attendances_type_2,a_e_attendances_other_a_e_department,a_e_attendances_booked_appointments_type_1,a_e_attendances_booked_appointments_type_2,a_e_attendances_booked_appointments_other_department,attendances_over_4hrs_type_1,attendances_over_4hrs_type_2,attendances_over_4hrs_other_department,attendances_over_4hrs_booked_appointments_type_1,attendances_over_4hrs_booked_appointments_type_2,attendances_over_4hrs_booked_appointments_other_department,patients_who_have_waited_4_12_hs_from_dta_to_admission,patients_who_have_waited_12_hrs_from_dta_to_admission,emergency_admissions_via_a_e_type_1,emergency_admissions_via_a_e_type_2,emergency_admissions_via_a_e_other_a_e_department,other_emergency_admissions,ingestion_timestamp,source_file,pipeline_version
MSitAE-JUNE-2025,RJ8,NHS ENGLAND SOUTH WEST,CORNWALL PARTNERSHIP NHS FOUNDATION TRUST,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16,78,2026-06-29T12:46:21.222Z,dbfs:/Workspace/Users/harakrishnashah@gmail.com/2025-06-ae-attendances.csv,1.0
MSitAE-JUNE-2025,RW1,NHS ENGLAND SOUTH EAST,HAMPSHIRE AND ISLE OF WIGHT HEALTHCARE NHS FOUNDATION TRUST,0,0,3100,0,0,0,0,0,52,0,0,0,0,0,0,0,4,120,2026-06-29T12:46:21.222Z,dbfs:/Workspace/Users/harakrishnashah@gmail.com/2025-06-ae-attendances.csv,1.0
MSitAE-JUNE-2025,Y02615,NHS ENGLAND MIDLANDS,SOUTH BIRMINGHAM GP WALK IN CENTRE,0,0,6125,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-06-29T12:46:21.222Z,dbfs:/Workspace/Users/harakrishnashah@gmail.com/2025-06-ae-attendances.csv,1.0
MSitAE-JUNE-2025,Y02676,NHS ENGLAND SOUTH EAST,BRIGHTON STATION HEALTH CENTRE,0,0,1793,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,2026-06-29T12:46:21.222Z,dbfs:/Workspace/Users/harakrishnashah@gmail.com/2025-06-ae-attendances.csv,1.0
MSitAE-JUNE-2025,AAH,NHS ENGLAND SOUTH WEST,TETBURY HOSPITAL TRUST LTD,0,0,611,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-06-29T12:46:21.222Z,dbfs:/Workspace/Users/harakrishnashah@gmail.com/2025-06-ae-attendances.csv,1.0



Bronze layer verified and complete!
